# PhysAR-IDS -- NB4c (v6): Attack 2 -- does the evasion survive when the attacker is confined to the observed feature range?

**Why.** NB4/NB4b let the agent push cyber features anywhere (in standardised units), including beyond the minimum / maximum seen in training. A tree ensemble extrapolates such values arbitrarily, so part of the evasion could be an artefact of out-of-range inputs rather than a weakness an attacker could exploit with real traffic. NB4b's ransomware / reverseshell flip (0.5 -> 1.0 sigma) coincided with 42% of rows being pushed >0.5 sigma beyond the training range. This check removes that freedom.

**Constraint.** After every step each controlled feature is clipped to `[min_train, max_train]` (a test value that already lies outside the range may stay where it is but may not move further out). Everything else is as in NB4 / NB4b: same black-box hard-label oracle, same 60 cyber features, same reward, same algorithm, same 5 seeds, full attack-class test set.

**Stages**
* **A (always, minutes):** re-score every saved agent (budgets 0.1, 0.2, 0.3, 0.5, 1.0 sigma and the unconstrained agent) with the clip switched on. This is a *lower bound*: the agents were not trained to cope with the clip.
* **B (`RETRAIN = True`, ~7 min per agent):** retrain PPO *with* the clip active, at budget 1.0 sigma and unconstrained (ceiling 3 sigma): 10 agents for 5 seeds (~70 min, resumable). This is the fair test. Set `RETRAIN = False` for stage A only.
* **C (seed 42):** a distribution check independent of the min/max rule: how far each adversarial cyber vector sits from its nearest training row (60-D, standardised), compared with clean test rows. Clipping can keep every feature inside its own range while the *combination* is still unlike any real traffic; this shows whether it is.

**Reading the result.** If ransomware / reverseshell evasion at 1.0 sigma (and unconstrained) stays high after retraining with the clip, the budget-threshold finding does not depend on out-of-range values. If it collapses, the finding must be reported as conditional on values the model has never seen. Neither outcome shows that a real network flow can produce the values; that would need packet-level construction and is stated as a limitation.

**Environment:** as NB4 (`stable-baselines3 sb3-contrib gymnasium` installed from a terminal).

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
CLIP_BUDGETS = {'1.0': 1.0, 'unconstrained': None}     # budgets at which agents are RETRAINED with range clipping (stage B)
RETRAIN = True                                          # False = stage A only (re-score saved agents with clipping; minutes)
SAVED_AGENTS = {'0.1': ('ppo_attack2_budgeted_{v}_seed{s}', 0.1), '0.2': ('ppo_attack2_budget0p2_{v}_seed{s}', 0.2), '0.3': ('ppo_attack2_budget0p3_{v}_seed{s}', 0.3),
                '0.5': ('ppo_attack2_budget0p5_{v}_seed{s}', 0.5), '1.0': ('ppo_attack2_budget1p0_{v}_seed{s}', 1.0), 'unconstrained': ('ppo_attack2_unconstrained_{v}_seed{s}', None)}
EVAL_CHUNK = 8192

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
import sb3_contrib, stable_baselines3
print(f'sb3-contrib {sb3_contrib.__version__} | stable-baselines3 {stable_baselines3.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data and the NB2 tuned configuration

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. Oracle, environment and batched evaluator (NB4 code + the range clip)

In [ ]:
class Oracle:
    """Hard-label black-box oracle over the fitted XGBoost classifier."""
    def __init__(self, clf): self.clf = clf; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X)
        return self.clf.predict(X)

def apply_action(x_cur, x_orig, action, eps_bound=EPS_BOUND, eps_total=None, range_clip=False):
    """In place on x_cur[:, CYBER_IDX]; returns the applied (post-clamp) perturbation."""
    a = np.clip(action, -eps_bound, eps_bound).astype(np.float32)
    old = x_cur[:, CYBER_IDX].copy(); prop = old + a
    if eps_total is not None:
        o = x_orig[:, CYBER_IDX]; prop = o + np.clip(prop - o, -eps_total, eps_total)
    if range_clip:   # never leave the observed training range [min, max] (a value already outside it may stay where it was, but may not go further out)
        prop = np.clip(prop, np.minimum(FEAT_LO[CYBER_IDX], x_orig[:, CYBER_IDX]), np.maximum(FEAT_HI[CYBER_IDX], x_orig[:, CYBER_IDX]))
    x_cur[:, CYBER_IDX] = prop
    return (prop - old).astype(np.float32)

def build_obs(x_cur, last_applied, last_pred, step):
    oh = np.zeros((len(x_cur), 6), dtype=np.float32); oh[np.arange(len(x_cur)), last_pred] = 1.0
    return np.concatenate([x_cur.astype(np.float32), last_applied.astype(np.float32), oh, np.full((len(x_cur), 1), step / MAX_STEPS, dtype=np.float32)], 1)

class IDSEvasionEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, X_pool, y_pool, oracle, eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None, eps_total_budget=None, range_clip=False):
        super().__init__(); self.range_clip = range_clip
        self.X_pool, self.y_pool, self.oracle = X_pool, y_pool, oracle
        self.eps_bound, self.max_steps, self.eps_total_budget = eps_bound, max_steps, eps_total_budget
        self.rng = np.random.RandomState(seed)
        self.action_space = spaces.Box(low=-eps_bound, high=eps_bound, shape=(N_CYBER,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(N_FEAT + N_CYBER + 6 + 1,), dtype=np.float32)
    def _obs(self): return build_obs(self.x_current[None], self.last_applied[None], np.array([self.last_pred]), self.step_count)[0]
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy(); self.x_current = self.x_orig.copy(); self.y_true = self.y_pool[idx]
        self.step_count = 0; self.last_applied = np.zeros(N_CYBER, dtype=np.float32)
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); return self._obs(), {}
    def step(self, action):
        applied = apply_action(self.x_current[None], self.x_orig[None], np.asarray(action, dtype=np.float32)[None], self.eps_bound, self.eps_total_budget, self.range_clip)[0]
        self.last_applied = applied; self.step_count += 1
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); pen = STEP_PENALTY_SCALE * float(np.sum(applied ** 2))
        terminated = truncated = False
        if self.last_pred == 0: reward = EVASION_REWARD - pen; terminated = True
        elif self.step_count >= self.max_steps: reward = TIMEOUT_PENALTY - pen; truncated = True
        else: reward = -pen
        return self._obs(), reward, terminated, truncated, {'evaded': self.last_pred == 0}

def run_batched(policy, X0, oracle, eps_total, mode='ppo', rng=None, range_clip=False):
    """Run one episode per row of X0 in parallel (10 steps max). mode: 'ppo' (deterministic policy) or 'random'."""
    N = len(X0); x_cur = X0.copy(); last_applied = np.zeros((N, N_CYBER), dtype=np.float32)
    last_pred = np.concatenate([oracle.predict(X0[i:i + 65536]) for i in range(0, N, 65536)]).astype(np.int64); clean_pred = last_pred.copy()
    active = np.ones(N, bool); evaded = np.zeros(N, bool); steps = np.zeros(N, np.int64)
    if mode == 'ppo': h = np.zeros((1, N, 64), np.float32); c = np.zeros((1, N, 64), np.float32)
    for step in range(MAX_STEPS):
        idx = np.where(active)[0]
        if len(idx) == 0: break
        obs = build_obs(x_cur[idx], last_applied[idx], last_pred[idx], step)
        if mode == 'ppo':
            act = np.zeros((len(idx), N_CYBER), np.float32)
            for s in range(0, len(idx), EVAL_CHUNK):
                sl = idx[s:s + EVAL_CHUNK]
                a, (hn, cn) = policy.predict(obs[s:s + EVAL_CHUNK], state=(h[:, sl], c[:, sl]), episode_start=np.full(len(sl), step == 0), deterministic=True)
                act[s:s + EVAL_CHUNK] = a; h[:, sl] = hn; c[:, sl] = cn
        else:
            act = rng.uniform(-EPS_BOUND, EPS_BOUND, size=(len(idx), N_CYBER)).astype(np.float32)
        sub = x_cur[idx]; applied = apply_action(sub, X0[idx], act, EPS_BOUND, eps_total, range_clip); x_cur[idx] = sub; last_applied[idx] = applied
        pred = np.concatenate([oracle.predict(sub[i:i + 65536]) for i in range(0, len(sub), 65536)]).astype(np.int64); last_pred[idx] = pred
        ev = pred == 0; evaded[idx[ev]] = True; steps[idx[ev]] = step + 1; active[idx[ev]] = False
    return {'evaded': evaded, 'steps': steps, 'x_final': x_cur, 'pred_final': last_pred, 'clean_pred': clean_pred}

def per_class(vec_bool):
    return np.array([vec_bool[y_atk_test == c].mean() for c in range(1, 6)])
def mean_steps(res):
    return np.array([res['steps'][(y_atk_test == c) & res['evaded']].mean() if ((y_atk_test == c) & res['evaded']).any() else np.nan for c in range(1, 6)])
def summarise(res):
    disp = np.abs(res['x_final'][:, CYBER_IDX] - X_atk_test[:, CYBER_IDX])
    _ex = lambda Z: np.maximum(np.maximum(FEAT_LO[CYBER_IDX] - Z[:, CYBER_IDX], Z[:, CYBER_IDX] - FEAT_HI[CYBER_IDX]), 0).max(1)
    exc = np.maximum(_ex(res['x_final']) - _ex(X_atk_test), 0)    # sigma ADDED beyond what the clean test row already had outside the training min/max
    return {'per_class_evasion': per_class(res['evaded']), 'overall_evasion': float(res['evaded'].mean()), 'mean_steps_to_evade': mean_steps(res),
            'mean_queries': float(1 + np.where(res['evaded'], res['steps'], MAX_STEPS).mean()),
            'attack_f1': float(f1_score(y_atk_test, res['pred_final'], average='weighted')),
            'max_abs_disp_mean': float(disp.max(1).mean()), 'max_abs_disp_max': float(disp.max()), 'mean_excess_beyond_train_range': float(exc.mean()), 'frac_rows_beyond_range_0.5sigma': float((exc > 0.5).mean()),
            'evaded': res['evaded'].copy()}
print('environment + evaluator defined; cyber features controlled:', N_CYBER)

## 3. Training / loading a clipped agent (resumable, fingerprinted)

In [ ]:
def fp_of(*parts): return hashlib.sha256(repr(parts).encode()).hexdigest()[:16]
def model_fp(oracle, X_probe): return hashlib.sha256(np.asarray(oracle.predict(X_probe)).tobytes()).hexdigest()[:16]
def latest_ckpt(d):
    if not os.path.isdir(d): return None
    c = [f for f in os.listdir(d) if f.endswith('_steps.zip')]
    if not c: return None
    best = max(c, key=lambda f: int(f.rsplit('_steps.zip', 1)[0].rsplit('_', 1)[1])); return os.path.join(d, best)

def get_agent(variant, seed, oracle):
    eps_total = CLIP_BUDGETS[variant]; name = f'ppo_attack2_clip_{variant.replace(".", "p")}_{PIPELINE_VERSION}_seed{seed}'
    fp = fp_of(model_fp(oracle, X_atk_train[:300]), EPS_BOUND, MAX_STEPS, EVASION_REWARD, STEP_PENALTY_SCALE, TIMEOUT_PENALTY, TOTAL_TIMESTEPS, eps_total, seed, N_CYBER, 'range_clip')
    if os.path.exists(name + '.zip') and os.path.exists(name + '.fp') and open(name + '.fp').read() == fp:
        print(f'    [{variant}/{seed}] finished agent found (fingerprint matches) -- loaded'); return RecurrentPPO.load(name), 0.0
    ckdir = f'_ckpt_{name}'; os.makedirs(ckdir, exist_ok=True); fpf = os.path.join(ckdir, '_fp.txt')
    env = IDSEvasionEnv(X_atk_train, y_atk_train, oracle, seed=seed, eps_total_budget=eps_total, range_clip=True)
    ck = latest_ckpt(ckdir)
    if ck and os.path.exists(fpf) and open(fpf).read() == fp:
        model = RecurrentPPO.load(ck, env=env); remaining = TOTAL_TIMESTEPS - model.num_timesteps
        print(f'    [{variant}/{seed}] resuming from {model.num_timesteps}/{TOTAL_TIMESTEPS}')
    else:
        shutil.rmtree(ckdir, ignore_errors=True); os.makedirs(ckdir); open(fpf, 'w').write(fp)
        model = RecurrentPPO('MlpLstmPolicy', env, n_steps=128, batch_size=64, learning_rate=3e-4, gamma=0.99, seed=seed, verbose=0,
                             policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1), device='cpu'); remaining = TOTAL_TIMESTEPS
    t0 = time.time()
    if remaining > 0:
        model.learn(total_timesteps=remaining, reset_num_timesteps=False, callback=CheckpointCallback(save_freq=CKPT_FREQ, save_path=ckdir, name_prefix='ckpt'), progress_bar=False)
    model.save(name); open(name + '.fp', 'w').write(fp); shutil.rmtree(ckdir, ignore_errors=True)
    return model, time.time() - t0

## 4. Per-seed run (stages A and B)

In [ ]:
if SMOKE_TEST: TOTAL_TIMESTEPS = 1024
R = {}; INFO = {}; ADV = {}; t_all = time.time()         # R[seed][key] -> summary
for seed in SEEDS:
    pf = f'attack2c_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = fp_of(best_params, EPS_BOUND, MAX_STEPS, TOTAL_TIMESTEPS, tuple(SAVED_AGENTS), tuple(CLIP_BUDGETS), RETRAIN, seed, X_atk_test.shape, float(X_atk_test[:5].sum()))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed] = saved['R']; INFO[seed] = saved['info']; ADV[seed] = saved.get('adv', {}); print(f'seed {seed}: result file found (fingerprint matches) -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False)
    f1_fit = f1_score(y_test, clf.predict(X_test), average='weighted'); clf.set_params(device='cpu', n_jobs=-1); oracle = Oracle(clf)
    INFO[seed] = {'clean_f1_fit_backend': float(f1_fit), 'clean_f1_cpu_oracle': float(f1_score(y_test, clf.predict(X_test), average='weighted'))}
    R[seed] = {}; ADV[seed] = {}
    # ---- stage A: saved agents, scored without and with the clip; random baseline with the clip
    for tag, (pat, budget) in SAVED_AGENTS.items():
        path = pat.format(v=PIPELINE_VERSION, s=seed) + '.zip'
        if not os.path.exists(path): print(f'  {path} not found -- skipped (run NB4 / NB4b first)'); continue
        agent = RecurrentPPO.load(path); r0 = run_batched(agent, X_atk_test, oracle, budget, mode='ppo'); r1 = run_batched(agent, X_atk_test, oracle, budget, mode='ppo', range_clip=True)
        R[seed][('saved_noclip', tag)] = summarise(r0); R[seed][('saved_clip', tag)] = summarise(r1)
        R[seed][('random_clip', tag)] = summarise(run_batched(None, X_atk_test, oracle, budget, mode='random', rng=np.random.RandomState(seed + 500), range_clip=True))
        if seed == SEEDS[0] and tag in ('1.0', 'unconstrained'): ADV[seed][('saved_noclip', tag)] = r0['x_final'].astype(np.float32); ADV[seed][('saved_clip', tag)] = r1['x_final'].astype(np.float32)
        print(f'  [A] {tag:>13}: unclipped overall {R[seed][("saved_noclip", tag)]["overall_evasion"]:.3f} (ransom {R[seed][("saved_noclip", tag)]["per_class_evasion"][3]:.3f}) -> clipped {R[seed][("saved_clip", tag)]["overall_evasion"]:.3f} (ransom {R[seed][("saved_clip", tag)]["per_class_evasion"][3]:.3f})', flush=True)
    # ---- stage B: retrain WITH the clip
    if RETRAIN:
        for tag, budget in CLIP_BUDGETS.items():
            agent, tt = get_agent(tag, seed, oracle); res = run_batched(agent, X_atk_test, oracle, budget, mode='ppo', range_clip=True); R[seed][('retrained_clip', tag)] = summarise(res)
            if seed == SEEDS[0]: ADV[seed][('retrained_clip', tag)] = res['x_final'].astype(np.float32)
            r = R[seed][('retrained_clip', tag)]
            print(f'  [B] {tag:>13}: retrained with clip ({tt / 60:.1f} min) -> per-class {np.round(r["per_class_evasion"], 3).tolist()} overall {r["overall_evasion"]:.3f} | queries {r["mean_queries"]:.2f}', flush=True)
    joblib.dump({'fp': cfg_fp, 'R': R[seed], 'info': INFO[seed], 'adv': ADV[seed]}, pf)
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 5. Results: per-class evasion without and with the clip (mean +/- std over seeds)

In [ ]:
atk_names = class_names[1:]
def have(key): return [s for s in SEEDS if key in R[s]]
def arr(key, field): return np.stack([np.asarray(R[s][key][field]) for s in have(key)])
sdv = lambda x: x.std(0, ddof=1) if len(x) > 1 else np.zeros(x.shape[1:])
rows = []
for tag in SAVED_AGENTS:
    for kind, label in [('saved_noclip', 'saved agent, unclipped (NB4/NB4b)'), ('saved_clip', 'saved agent, clip applied at test'), ('random_clip', 'random search, clipped'), ('retrained_clip', 'RETRAINED with clip')]:
        key = (kind, tag)
        if not have(key): continue
        a = arr(key, 'per_class_evasion'); o = arr(key, 'overall_evasion'); n = len(have(key))
        row = {'budget': tag, 'agent / attack': label, 'seeds': n}; row.update({c: f'{a[:, i].mean():.3f} +/- {sdv(a)[i]:.3f}' for i, c in enumerate(atk_names)})
        row['ALL'] = f'{o.mean():.3f} +/- {(o.std(ddof=1) if n > 1 else 0):.3f}'; row['queries'] = f'{arr(key, "mean_queries").mean():.2f}'; row['mean added excess (sigma)'] = f'{arr(key, "mean_excess_beyond_train_range").mean():.3f}'; rows.append(row)
clip_table = pd.DataFrame(rows); pd.set_option('display.width', 280); print(clip_table.to_string(index=False))

print('\nPer-seed ransomware / reverseshell evasion (the classes whose flip is in question):')
for tag in ['0.5', '1.0', 'unconstrained']:
    for kind in ['saved_noclip', 'saved_clip', 'retrained_clip']:
        key = (kind, tag)
        if have(key): print(f'  {tag:>13} {kind:<15} ransomware {np.round(arr(key, "per_class_evasion")[:, 3], 3).tolist()}  reverseshell {np.round(arr(key, "per_class_evasion")[:, 4], 3).tolist()}')

## 6. Paired tests (same target per seed; n = 5, Holm-adjusted over the 5 classes)

Does clipping change the result? Retrained-with-clip versus the unclipped agent at the same budget, and retrained-with-clip versus clipped random search.

In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
def ptest(A, B):
    d = A - B
    if len(d) < 2: return float('nan')
    if d.std(ddof=1) == 0: return 1.0 if d.mean() == 0 else 0.0
    return float(stats.ttest_rel(A, B).pvalue)
stat_rows = []
for tag in CLIP_BUDGETS if RETRAIN else []:
    for ref_kind, label in [('saved_noclip', 'unclipped agent'), ('random_clip', 'random (clipped)')]:
        ks = [('retrained_clip', tag), (ref_kind, tag)]
        if not all(len(have(k)) == len(SEEDS) for k in ks): continue
        A, B = arr(ks[0], 'per_class_evasion'), arr(ks[1], 'per_class_evasion'); raw = np.array([ptest(A[:, i], B[:, i]) for i in range(5)]); adj = holm(np.nan_to_num(raw, nan=1.0))
        oa, ob = arr(ks[0], 'overall_evasion'), arr(ks[1], 'overall_evasion')
        for i, c in enumerate(atk_names): stat_rows.append({'budget': tag, 'retrained-clip minus': label, 'class': c, 'diff': f'{(A - B)[:, i].mean():+.3f}', 'p_raw': f'{raw[i]:.4f}', 'p_holm': f'{adj[i]:.4f}'})
        stat_rows.append({'budget': tag, 'retrained-clip minus': label, 'class': 'ALL', 'diff': f'{(oa - ob).mean():+.3f}', 'p_raw': f'{ptest(oa, ob):.4f}', 'p_holm': '-'})
stat_table = pd.DataFrame(stat_rows); print(stat_table.to_string(index=False) if len(stat_table) else 'no paired tests (RETRAIN=False or incomplete seeds)')

## 7. Stage C -- do the evading rows look like real benign traffic? (seed 42)

Staying inside every feature's min/max does not make a 60-feature vector realistic: the *combination* can still be unlike any real traffic. Two distances, in the 60 cyber features (standardised), against a 100,000-row training sample:

* **distance to the nearest *benign* training row, for the rows that evaded** (the attacker's goal is to look benign). Reference: the same distance for *genuine benign test rows* (their natural spread). The share of evading rows inside the genuine benign 95th percentile says how many are plausible benign mimics; the rest sit in regions the benign class never occupies and are classified benign only because the trees extrapolate there.
* distance to the nearest training row of *any* class, for all adversarial rows vs the clean attack rows they came from (how far the attack moved the row from the data manifold).

Class-balanced sample of up to 600 attack test rows per class. A mimic share near 100% would mean the evasion lands in real benign territory; near 0% would mean it is an extrapolation artefact.

In [ ]:
def nn_dist(A, ref, chunk=500):
    out = []; ref_t = torch.from_numpy(ref).float().to(DEVICE)
    for i in range(0, len(A), chunk): out.append(torch.cdist(torch.from_numpy(A[i:i + chunk]).float().to(DEVICE), ref_t).min(1).values.cpu().numpy())
    return np.concatenate(out)
rng = np.random.RandomState(7)
ref_any = X_train[rng.choice(len(X_train), size=min(100000, len(X_train)), replace=False)][:, CYBER_IDX]
ben_tr = X_train[y_train == 0]; ref_ben = ben_tr[rng.choice(len(ben_tr), size=min(100000, len(ben_tr)), replace=False)][:, CYBER_IDX]
sel = np.concatenate([rng.choice(np.where(y_atk_test == c)[0], size=min(600, (y_atk_test == c).sum()), replace=False) for c in range(1, 6)])
ben_te = X_test[y_test == 0]; ben_te = ben_te[rng.choice(len(ben_te), size=min(3000, len(ben_te)), replace=False)][:, CYBER_IDX]
d_ben_natural = nn_dist(ben_te, ref_ben); thr_ben = np.percentile(d_ben_natural, 95)
d_any_clean = nn_dist(X_atk_test[sel][:, CYBER_IDX], ref_any)
print(f'Genuine benign test rows -> nearest benign training row: median {np.median(d_ben_natural):.3f}, 95th percentile {thr_ben:.3f}')
nn_rows = []; cl = y_atk_test[sel]; s0 = SEEDS[0]
for key, X_adv in ADV.get(s0, {}).items():
    ev = R[s0][key]['evaded'][sel]; Xs = X_adv[sel][:, CYBER_IDX]
    d_any = nn_dist(Xs, ref_any); d_ben = nn_dist(Xs[ev], ref_ben) if ev.any() else np.array([np.nan])
    row = {'adversarial set': f'{key[0]} / {key[1]}', 'evaded rows (sample)': int(ev.sum()), 'median dist to nearest benign (evaded)': f'{np.nanmedian(d_ben):.3f}',
           'share of evaded rows within genuine-benign 95th pct': f'{np.mean(d_ben <= thr_ben):.3f}', 'median dist to nearest any-class row': f'{np.median(d_any):.3f}'}
    for c in (4, 5):
        m = ev & (cl == c); row[f'{atk_names[c - 1]}: benign-mimic share'] = f'{np.mean(nn_dist(Xs[m], ref_ben) <= thr_ben):.3f}' if m.sum() > 0 else 'n/a (none evaded)'
    nn_rows.append(row)
nn_rows.insert(0, {'adversarial set': 'clean attack rows (reference)', 'evaded rows (sample)': '-', 'median dist to nearest benign (evaded)': '-', 'share of evaded rows within genuine-benign 95th pct': '-', 'median dist to nearest any-class row': f'{np.median(d_any_clean):.3f}'})
nn_table = pd.DataFrame(nn_rows); print(nn_table.T.to_string(header=False))

## 8. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8)); tags = list(SAVED_AGENTS); w = 0.2
for k, (kind, label) in enumerate([('saved_noclip', 'saved, unclipped'), ('saved_clip', 'saved, clip at test'), ('retrained_clip', 'retrained with clip'), ('random_clip', 'random, clipped')]):
    ys = [arr((kind, t), 'overall_evasion').mean() if have((kind, t)) else np.nan for t in tags]; es = [arr((kind, t), 'overall_evasion').std(ddof=1) if len(have((kind, t))) > 1 else 0 for t in tags]
    ax[0].bar(np.arange(len(tags)) + (k - 1.5) * w, ys, w, yerr=es, capsize=2, label=label)
    ys = [arr((kind, t), 'per_class_evasion')[:, 3:].mean() if have((kind, t)) else np.nan for t in tags]; ax[1].bar(np.arange(len(tags)) + (k - 1.5) * w, ys, w, label=label)
for a, t in [(ax[0], 'Overall evasion'), (ax[1], 'Ransomware + reverseshell evasion (mean)')]:
    a.set_xticks(np.arange(len(tags))); a.set_xticklabels(tags); a.set_xlabel('cumulative budget (sigma)'); a.set_title(t); a.grid(alpha=.3, axis='y'); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'attack2c_range_clip_{PIPELINE_VERSION}.png', dpi=150); plt.show()

joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'RETRAIN': RETRAIN, 'class_names': class_names, 'clip_table': clip_table, 'stat_table': stat_table, 'nn_table': nn_table,
             'per_seed': {s: {f'{k[0]}|{k[1]}': {kk: vv for kk, vv in v.items() if kk != 'evaded'} for k, v in R[s].items()} for s in SEEDS}, 'info': INFO},
            f'attack2c_results_{PIPELINE_VERSION}.pkl')
clip_table.to_csv(f'attack2c_clip_table_{PIPELINE_VERSION}.csv', index=False); stat_table.to_csv(f'attack2c_paired_tests_{PIPELINE_VERSION}.csv', index=False); nn_table.to_csv(f'attack2c_nn_distance_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB4c', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'retrain': RETRAIN, 'timesteps': TOTAL_TIMESTEPS, 'sb3_contrib': sb3_contrib.__version__, 'torch': torch.__version__,
           'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb4c_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack2c_results_{PIPELINE_VERSION}.pkl', f'attack2c_clip_table_{PIPELINE_VERSION}.csv', f'attack2c_paired_tests_{PIPELINE_VERSION}.csv', f'attack2c_nn_distance_{PIPELINE_VERSION}.csv', f'attack2c_range_clip_{PIPELINE_VERSION}.png', f'nb4c_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)